<a href="https://colab.research.google.com/github/xiaosu/Cool/blob/master/demo1_embedding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [14]:
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Source Text (Your data)
raw_text = """
Bob likes computers.
Bob likes bananas.
Bob likes cars.
"""

# 2. Chunking: Splitting text into smaller, meaningful pieces
text_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=10,
    chunk_overlap=2
)
docs = text_splitter.split_text(raw_text)
print(f"✅ Created {len(docs)} chunks.")

# 3. Embedding: Converting text into mathematical vectors
# We use a popular, lightweight model from HuggingFace
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 4. Storage: Create an In-Memory Vector Database (FAISS)
vector_db = FAISS.from_texts(docs, embeddings)
print("✅ Vector database is ready in memory.")

# 5. Query: Finding information using 'Similarity Search'
query = "What fruit does Bob like?"
search_results = vector_db.similarity_search(query, k=3)

print("\n--- Search Result ---")
print(f"Top Match: {search_results[0].page_content}")


✅ Created 3 chunks.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Vector database is ready in memory.

--- Search Result ---
Top Match: Bob likes bananas.


In [15]:
import numpy as np

# 1. Access the underlying FAISS index
index = vector_db.index

# 2. Extract all vectors back into a NumPy array
# index.ntotal gives the number of chunks stored
# index.d gives the number of dimensions (384 for this model)
all_vectors = index.reconstruct_n(0, index.ntotal)

print(f"Total Chunks Stored: {index.ntotal}")
print(f"Dimensions per Vector: {index.d}")
print("-" * 30)

# 3. View the details of the first chunk's array
for i, chunk_text in enumerate(docs):
    print(f"Chunk {i}: {chunk_text[:50]}...")
    print(f"Array (first 50 numbers): {all_vectors[i][:50]}")
    print(f"Array Shape: {all_vectors[i].shape}")
    print("-" * 30)


Total Chunks Stored: 3
Dimensions per Vector: 384
------------------------------
Chunk 0: Bob likes computers....
Array (first 50 numbers): [ 0.0217791   0.05715422 -0.03383412 -0.12839507 -0.07887006 -0.0446372
  0.09925419  0.01280197  0.00604287  0.03795321  0.02506599  0.06900605
  0.05261324 -0.0458772   0.08394896  0.02940306  0.00096607 -0.07447337
  0.01065818 -0.03631914 -0.03934449 -0.02523015  0.0085178  -0.09070157
  0.00180928  0.08699682  0.03530988 -0.0730792  -0.06135365 -0.03181315
 -0.04088674  0.01759054 -0.00672754  0.01339317 -0.02305891 -0.01671073
 -0.00611552  0.04218307 -0.06834969 -0.0600545  -0.07796076 -0.09676401
  0.06804022  0.07868378 -0.0651836  -0.00250089  0.01575415 -0.01977837
  0.117544   -0.02989245]
Array Shape: (384,)
------------------------------
Chunk 1: Bob likes bananas....
Array (first 50 numbers): [-0.0019394  -0.00640576 -0.02590472 -0.03561325 -0.03336533  0.01975928
  0.09796604  0.0207286  -0.0323929   0.08911285  0.01987795 -0.072659